# RDPro NLP To Scala Via Python Raw

Start from free-text NLP, generate a raw Python reference, and pass that raw Python into Scala generation as the ablation path.


In [18]:
from pathlib import Path
import json
import sys
import importlib

PROJECT_ROOT = Path('/Users/clockorangezoe/Documents/phd_projects/code/geoAI/LangGraph').resolve()
NOTEBOOK_DIR = PROJECT_ROOT / 'Notebook'
CODEGEN_DIR = NOTEBOOK_DIR / 'rdpro_section_codegen'
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

for mod in [
    'rdpro_section_codegen',
    'rdpro_section_codegen.langgraph_section_agent',
]:
    if mod in sys.modules:
        del sys.modules[mod]

import rdpro_section_codegen
import rdpro_section_codegen.langgraph_section_agent

importlib.reload(rdpro_section_codegen.langgraph_section_agent)
importlib.reload(rdpro_section_codegen)

from rdpro_section_codegen.langgraph_section_agent import _generate_python_reference



In [19]:
FREE_TEXT_TASK = """
Calculate land-use category percentages and summary statistics for Boston neighborhoods from NLCD raster data.
Input raster path: /Users/clockorangezoe/Downloads/Annual_NLCD_LndCov_2024_CU_C1V1/Annual_NLCD_LndCov_2024_CU_C_1V1_clip.tif.
Vector neighborhood polygon path: /Users/clockorangezoe/Downloads/boston_neighborhood_boundaries/Boston_Neighborhood_Boundaries_sample.shp.
This is a per-polygon zonal workflow.
Produce tabular CSV outputs with per-neighborhood class percentages and a neighborhood-level dominant class summary.
""".strip()

TRANSLATION_MODE = 'via-python-raw'  # one of: 'direct', 'via-python', 'via-python-raw'
PROVIDER = 'openai'
MODEL = 'gpt-4o'

print(FREE_TEXT_TASK)
print('translation_mode =', TRANSLATION_MODE)


Calculate land-use category percentages and summary statistics for Boston neighborhoods from NLCD raster data.
Input raster path: /Users/clockorangezoe/Downloads/Annual_NLCD_LndCov_2024_CU_C1V1/Annual_NLCD_LndCov_2024_CU_C_1V1_clip.tif.
Vector neighborhood polygon path: /Users/clockorangezoe/Downloads/boston_neighborhood_boundaries/Boston_Neighborhood_Boundaries_sample.shp.
This is a per-polygon zonal workflow.
Produce tabular CSV outputs with per-neighborhood class percentages and a neighborhood-level dominant class summary.
translation_mode = via-python-raw


In [20]:
cfg = {
    'provider': PROVIDER,
    'model': MODEL,
}

python_reference = _generate_python_reference(FREE_TEXT_TASK, cfg)
print('Generated Python reference:')
print(python_reference)
print('
via-python-raw notebook mode: raw Python is shown as translation context only.')
print('No notebook-side analyze_python_script(...) or build_plan(...) is used in this mode.')



LLM task info: {'task_type': 'zonal_stats', 'task_label': 'landcover', 'reason': 'The task involves calculating land-use category percentages and summary statistics for specific polygons (Boston neighborhoods) using NLCD raster data, which is a typical zonal statistics workflow.'}
Generated Python reference:
python
import rasterio
import geopandas as gpd
import numpy as np
import pandas as pd
from rasterio.mask import mask
from collections import Counter

# File paths
raster_path = '/Users/clockorangezoe/Downloads/Annual_NLCD_LndCov_2024_CU_C1V1/Annual_NLCD_LndCov_2024_CU_C_1V1_clip.tif'
vector_path = '/Users/clockorangezoe/Downloads/boston_neighborhood_boundaries/Boston_Neighborhood_Boundaries_sample.shp'
output_csv_path = '/Users/clockorangezoe/Downloads/neighborhood_land_use_summary.csv'

def calculate_land_use_percentages(raster_path, vector_path):
    # Load the vector data
    neighborhoods = gpd.read_file(vector_path)

    # Open the raster data
    with rasterio.open(raster_pat

## Run Section Agent

Launch the section-by-section generate/compile/fix loop in `via-python-raw` mode.

This notebook intentionally does **not** analyze the generated Python into task structure.
The generated Python is used as raw translation context only.



In [21]:
import json
import subprocess

ENV_PYTHON = Path('/Users/clockorangezoe/miniconda3/envs/geo_llm_spark/bin/python')
SECTION_AGENT = CODEGEN_DIR / 'langgraph_section_agent.py'
SCAFFOLD_SCALA = CODEGEN_DIR / 'job_scaffold.scala'
OUTPUT_SECTIONAL = CODEGEN_DIR / f'nlp_to_scala_{TRANSLATION_MODE}.scala'
WORK_DIR = CODEGEN_DIR / 'generated_output' / f'nlp_to_scala_{TRANSLATION_MODE}'
API_DOC = PROJECT_ROOT / 'Doc' / 'rdpro_api_doc_combined.md'
GUIDE = NOTEBOOK_DIR / 'RDProAgentLoop_perAPI_fix_migration_guide.md'

WORK_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    str(ENV_PYTHON),
    str(SECTION_AGENT),
    '--translation-mode', TRANSLATION_MODE,
    '--free-text', FREE_TEXT_TASK,
    '--scaffold', str(SCAFFOLD_SCALA),
    '--output-scala', str(OUTPUT_SECTIONAL),
    '--work-dir', str(WORK_DIR),
    '--api-doc', str(API_DOC),
    '--guide', str(GUIDE),
    '--provider', PROVIDER,
    '--model', MODEL,
]

print('Running command:')
print(' '.join(cmd))
result = subprocess.run(cmd, cwd=str(CODEGEN_DIR), capture_output=True, text=True)
print('returncode =', result.returncode)

summary = {}
stdout_text = (result.stdout or '').strip()
if stdout_text:
    try:
        summary = json.loads(stdout_text)
    except Exception:
        print('stdout was not valid JSON')
        print(stdout_text[:2000])

if summary:
    print(json.dumps({
        'success': summary.get('success'),
        'fail_reason': summary.get('fail_reason'),
        'input_mode': summary.get('input_mode'),
        'generated_python_reference': summary.get('generated_python_reference'),
        'planned_sections': summary.get('planned_sections'),
        'completed_sections': summary.get('completed_sections'),
        'output_scala': summary.get('output_scala'),
        'last_report': summary.get('last_report'),
        'last_merged': summary.get('last_merged'),
    }, indent=2))

if result.stderr:
    print('----- stderr -----')
    print(result.stderr[:4000])

print('Scala output exists =', OUTPUT_SECTIONAL.exists())



Running command:
/Users/clockorangezoe/miniconda3/envs/geo_llm_spark/bin/python /Users/clockorangezoe/Documents/phd_projects/code/geoAI/LangGraph/Notebook/rdpro_section_codegen/langgraph_section_agent.py --translation-mode via-python-raw --free-text Calculate land-use category percentages and summary statistics for Boston neighborhoods from NLCD raster data.
Input raster path: /Users/clockorangezoe/Downloads/Annual_NLCD_LndCov_2024_CU_C1V1/Annual_NLCD_LndCov_2024_CU_C_1V1_clip.tif.
Vector neighborhood polygon path: /Users/clockorangezoe/Downloads/boston_neighborhood_boundaries/Boston_Neighborhood_Boundaries_sample.shp.
This is a per-polygon zonal workflow.
Produce tabular CSV outputs with per-neighborhood class percentages and a neighborhood-level dominant class summary. --scaffold /Users/clockorangezoe/Documents/phd_projects/code/geoAI/LangGraph/Notebook/rdpro_section_codegen/job_scaffold.scala --output-scala /Users/clockorangezoe/Documents/phd_projects/code/geoAI/LangGraph/Notebook/r

returncode = 0
----- stdout -----
{
  "success": false,
  "done": true,
  "input_mode": "via-python-raw",
  "generated_python_reference": "/Users/clockorangezoe/Documents/phd_projects/code/geoAI/LangGraph/Notebook/rdpro_section_codegen/generated_output/nlp_to_scala_via-python-raw/generated_python_reference_20260322_134309_616936.py",
  "section_idx": 1,
  "planned_sections": [
    "LOAD_DATA",
    "TYPE_CHECK",
    "RASTER_VECTOR_JOIN",
    "ANALYTICS"
  ],
  "completed_sections": [
    "LOAD_DATA"
  ],
  "fail_reason": "Section TYPE_CHECK failed after 5 attempts",
  "output_scala": "/Users/clockorangezoe/Documents/phd_projects/code/geoAI/LangGraph/Notebook/rdpro_section_codegen/nlp_to_scala_via-python-raw.scala",
  "last_report": "/Users/clockorangezoe/Documents/phd_projects/code/geoAI/LangGraph/Notebook/rdpro_section_codegen/generated_output/nlp_to_scala_via-python-raw/run_TYPE_CHECK_t7_20260322_134426_374607/report.log",
  "last_merged": "/Users/clockorangezoe/Documents/phd_projects